## Feature Engineering — Climate Risk Model Testing

This notebook follows the original model-testing scaffold using the supplied `train_toy2_risk.csv`. It compares a property-only random forest with versions that add available flood, wildfire, and other environmental features.

The toy file contains 1,000 rows, all from January 2024. A month-based train/test split cannot be formed from this sample, so this version uses a reproducible **60% train / 20% validation / 20% test** split. Every model uses the same rows in each split. The validation set is used for comparison and model selection; the test set is held out for final scoring.

The attached `modeling.py` and `splits.py` are time-split helpers that depend on a missing `dataset_a` module. The workflow below keeps their core ideas—fit preprocessing only on training rows, fit a fresh model for each comparison, and report comparable scores—while remaining self-contained for this toy file. These random-split results are for pipeline development; a later full-data analysis should also use a spatial or time-based holdout if the intended use is future or out-of-area pricing.

### 1. Start with the Random Forest
Load libraries and the cleaned property/risk dataset. The target is `ClosePrice`. Derived target columns, raw response metadata, and pre-encoded location columns are excluded to avoid target leakage or dependence on preprocessing performed before the split.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.float_format", "{:,.4f}".format)
RANDOM_STATE = 42
TARGET = "ClosePrice"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("train_toy2_risk.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("upload/train_toy2_risk.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place train_toy2_risk.csv next to this notebook or in an upload/ folder."
    )

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise KeyError(f"Required target column {TARGET!r} was not found.")

# Keep only rows with a usable positive sale price. Positive prices are needed for MAPE.
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.loc[df[TARGET].gt(0)].copy()

# Use sale year/month as timing features when available; do not use the day of sale.
if "CloseDate" in df.columns:
    close_date = pd.to_datetime(df["CloseDate"], errors="coerce")
    df["sale_year"] = close_date.dt.year
    df["sale_month"] = close_date.dt.month

# Treat postal codes as categories rather than ordered numeric values.
if "PostalCode" in df.columns:
    df["PostalCode"] = df["PostalCode"].astype("object")

print(f"Rows available: {len(df):,}")
print(f"CloseDate range: {df['CloseDate'].min()} to {df['CloseDate'].max()}" if "CloseDate" in df else "CloseDate unavailable")
print(f"Source file: {DATA_PATH}")

### 2. Create comparable models for testing

The base group uses ordinary property attributes, location, and sale timing. Risk groups are built only from fields present in this file. Raw API JSON, retrieval timestamps, match counts, and status fields are left out because they describe acquisition/coverage rather than a property hazard measurement.

This sample has FEMA flood fields, CAL FIRE LRA/SRA hazard fields, and elevation. It does **not** include First Street Flood Factor, historical fire exposure, or insurance-market features. The corresponding model groups are omitted when no usable columns exist.

In [ ]:
BASE_CANDIDATES = [
    # Numeric property attributes
    "AttachedGarageYN", "BathroomsTotalInteger", "BedroomsTotal", "FireplaceYN",
    "GarageSpaces", "Latitude", "Longitude", "LivingArea", "LotSizeSquareFeet",
    "NewConstructionYN", "ParkingTotal", "PoolPrivateYN", "Stories", "ViewYN",
    "YearBuilt", "sale_year", "sale_month",
    # Categorical location/property attributes
    "City", "CountyOrParish", "Flooring", "Levels", "PostalCode",
    "HighSchoolDistrict",
]
FLOOD_CANDIDATES = ["fema_sfha", "fema_fld_zone", "fema_zone_subty"]
FIRE_CANDIDATES = ["fire_lra_fhsz_description", "fire_sra_fhsz_description"]
OTHER_ENVIRONMENT_CANDIDATES = ["elevation_m"]

# Match insurance columns by name so this notebook will include them when supplied later.
INSURANCE_CANDIDATES = [
    c for c in df.columns
    if any(term in c.lower() for term in ("insurance", "insured", "coverage_amount", "nonrenewal"))
]

def available_features(candidates):
    # Return present columns with at least one observed value.
    return [c for c in candidates if c in df.columns and df[c].notna().any()]

BASE_FEATURES = available_features(BASE_CANDIDATES)
FLOOD_FEATURES = available_features(FLOOD_CANDIDATES)
FIRE_FEATURES = available_features(FIRE_CANDIDATES)
OTHER_ENV_FEATURES = available_features(OTHER_ENVIRONMENT_CANDIDATES)
INSURANCE_FEATURES = available_features(INSURANCE_CANDIDATES)

def unique(items):
    return list(dict.fromkeys(items))

FEATURE_SETS = {"Random Forest — Baseline": BASE_FEATURES}
if FLOOD_FEATURES:
    FEATURE_SETS["Random Forest + Flood"] = unique(BASE_FEATURES + FLOOD_FEATURES)
if FIRE_FEATURES:
    FEATURE_SETS["Random Forest + Fire"] = unique(BASE_FEATURES + FIRE_FEATURES)
if FLOOD_FEATURES or FIRE_FEATURES or OTHER_ENV_FEATURES:
    FEATURE_SETS["Random Forest + Combined"] = unique(
        BASE_FEATURES + FLOOD_FEATURES + FIRE_FEATURES + OTHER_ENV_FEATURES
    )
if INSURANCE_FEATURES:
    FEATURE_SETS["Random Forest + Insurance"] = unique(
        BASE_FEATURES + FLOOD_FEATURES + FIRE_FEATURES + OTHER_ENV_FEATURES + INSURANCE_FEATURES
    )

if not BASE_FEATURES:
    raise ValueError("No usable baseline features were found. Check the input columns.")

print("Feature counts by model:")
for name, features in FEATURE_SETS.items():
    print(f"  {name}: {len(features)} columns")
print("\nAvailable risk features:")
print({
    "flood": FLOOD_FEATURES,
    "fire": FIRE_FEATURES,
    "other_environment": OTHER_ENV_FEATURES,
    "insurance": INSURANCE_FEATURES,
})
if not INSURANCE_FEATURES:
    print("Insurance model skipped: no insurance-market columns are present in this toy file.")

In [ ]:
def build_model(features):
    # Create a leakage-safe preprocessing + Random Forest pipeline.
    numeric_features = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    categorical_features = [c for c in features if c not in numeric_features]

    transformers = []
    if numeric_features:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        transformers.append(("numeric", numeric_pipeline, numeric_features))
    if categorical_features:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    forest = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        max_features=1 / 3,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", forest),
    ])


def regression_scores(y_true, y_pred):
    # Return comparable price-scale metrics; MAPE is expressed as percent.
    return {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mape_pct": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }


def fit_and_score(features, train, evaluation):
    # Fit a fresh pipeline on train rows and score it on evaluation rows.
    model = build_model(features)
    model.fit(train[features], train[TARGET])
    predictions = model.predict(evaluation[features])
    scores = regression_scores(evaluation[TARGET], predictions)
    return model, predictions, scores


### 3. Scoring
Use one reproducible split for every feature set. Compare validation MAE, RMSE, and R², then refit each candidate on train + validation and score once on the untouched test set. County/parish MAE is also reported for test predictions.

In [ ]:
train_df, holdout_df = train_test_split(
    df, test_size=0.40, random_state=RANDOM_STATE
)
validation_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=RANDOM_STATE
)

print(f"Train:      {len(train_df):,} rows")
print(f"Validation: {len(validation_df):,} rows")
print(f"Test:       {len(test_df):,} rows")

validation_rows = []
validation_fitted = {}
for name, features in FEATURE_SETS.items():
    model, predictions, scores = fit_and_score(features, train_df, validation_df)
    validation_fitted[name] = model
    validation_rows.append({"model": name, "n_features": len(features), **{
        f"validation_{metric}": value for metric, value in scores.items()
    }})

validation_results = pd.DataFrame(validation_rows).sort_values(
    "validation_r2", ascending=False, ignore_index=True
)
best_model_name = validation_results.loc[0, "model"]
print(f"Best validation R²: {best_model_name}")
display(validation_results)

# Evaluate every model on the same test rows, fitting preprocessing on train + validation only.
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)
test_rows = []
test_predictions = {}
final_fitted = {}
for name, features in FEATURE_SETS.items():
    model, predictions, scores = fit_and_score(features, train_validation_df, test_df)
    final_fitted[name] = model
    test_predictions[name] = predictions
    test_rows.append({"model": name, "n_features": len(features), **{
        f"test_{metric}": value for metric, value in scores.items()
    }})

test_results = pd.DataFrame(test_rows)
comparison = validation_results.merge(test_results, on=["model", "n_features"])
baseline_test_r2 = comparison.loc[
    comparison["model"] == "Random Forest — Baseline", "test_r2"
].iloc[0]
comparison["test_r2_vs_baseline"] = comparison["test_r2"] - baseline_test_r2
comparison = comparison.sort_values("validation_r2", ascending=False, ignore_index=True)
display(comparison)

In [ ]:
# Test-set MAE by county/parish for each candidate model.
county_column = "CountyOrParish"
county_mae_rows = []
if county_column in test_df.columns:
    for name, predictions in test_predictions.items():
        county_frame = pd.DataFrame({
            "county_or_parish": test_df[county_column].to_numpy(),
            "actual": test_df[TARGET].to_numpy(),
            "predicted": predictions,
        })
        county_frame["absolute_error"] = (county_frame["actual"] - county_frame["predicted"]).abs()
        grouped = county_frame.groupby("county_or_parish", dropna=False).agg(
            n=("absolute_error", "size"), county_mae=("absolute_error", "mean")
        ).reset_index()
        grouped.insert(0, "model", name)
        county_mae_rows.append(grouped)
    county_mae_test = pd.concat(county_mae_rows, ignore_index=True)
    display(county_mae_test.sort_values(["model", "county_mae"]))
else:
    county_mae_test = pd.DataFrame()
    print("County/parish MAE skipped: CountyOrParish column is not available.")

### 4. Permutation and Ablation Testing
Permutation importance uses held-out test rows and the validation-selected model. It measures the change in test MAE when one raw feature is shuffled; correlated features can share or mask importance, so treat the ranking as diagnostic.

The ablation table removes one risk group at a time from the combined feature set and compares validation scores. This helps identify whether each group adds signal beyond the other available risk groups without repeatedly consulting the final test set.

In [ ]:
# Permutation importance for the model selected using validation R².
best_features = FEATURE_SETS[best_model_name]
best_test_model = final_fitted[best_model_name]
perm = permutation_importance(
    best_test_model,
    test_df[best_features],
    test_df[TARGET],
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
permutation_results = pd.DataFrame({
    "feature": best_features,
    "mae_increase_mean": perm.importances_mean,
    "mae_increase_std": perm.importances_std,
}).sort_values("mae_increase_mean", ascending=False, ignore_index=True)
display(permutation_results)

ax = permutation_results.head(15).sort_values("mae_increase_mean").plot.barh(
    x="feature", y="mae_increase_mean", xerr="mae_increase_std",
    figsize=(9, 6), legend=False, color="#2a78d6",
)
ax.set_xlabel("Increase in test MAE when shuffled ($)")
ax.set_ylabel("")
ax.set_title(f"Permutation importance — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Validation-only feature-group ablation.
ABLATION_SETS = {}
if FLOOD_FEATURES and FIRE_FEATURES:
    combined_features = unique(BASE_FEATURES + FLOOD_FEATURES + FIRE_FEATURES + OTHER_ENV_FEATURES)
    ABLATION_SETS["Combined"] = combined_features
    ABLATION_SETS["Combined without flood"] = unique(BASE_FEATURES + FIRE_FEATURES + OTHER_ENV_FEATURES)
    ABLATION_SETS["Combined without fire"] = unique(BASE_FEATURES + FLOOD_FEATURES + OTHER_ENV_FEATURES)
    if OTHER_ENV_FEATURES:
        ABLATION_SETS["Combined without elevation"] = unique(BASE_FEATURES + FLOOD_FEATURES + FIRE_FEATURES)

if ABLATION_SETS:
    ablation_rows = []
    for name, features in ABLATION_SETS.items():
        _, _, scores = fit_and_score(features, train_df, validation_df)
        ablation_rows.append({"ablation": name, "n_features": len(features), **{
            f"validation_{metric}": value for metric, value in scores.items()
        }})
    ablation_validation = pd.DataFrame(ablation_rows)
    combined_r2 = ablation_validation.loc[
        ablation_validation["ablation"] == "Combined", "validation_r2"
    ].iloc[0]
    combined_mae = ablation_validation.loc[
        ablation_validation["ablation"] == "Combined", "validation_mae"
    ].iloc[0]
    ablation_validation["r2_change_vs_combined"] = ablation_validation["validation_r2"] - combined_r2
    ablation_validation["mae_change_vs_combined"] = ablation_validation["validation_mae"] - combined_mae
    display(ablation_validation)
else:
    ablation_validation = pd.DataFrame()
    print("Group ablation skipped: both flood and fire feature groups are needed for this comparison.")

### 5. Results
Save the model comparison, county-level errors, permutation importance, and validation ablation tables. The CSVs are written to `results/` relative to the notebook’s working directory.

In [ ]:
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
county_mae_test.to_csv(RESULTS_DIR / "county_mae_test.csv", index=False)
permutation_results.to_csv(RESULTS_DIR / "permutation_importance.csv", index=False)
if not ablation_validation.empty:
    ablation_validation.to_csv(RESULTS_DIR / "ablation_validation.csv", index=False)

print(f"Saved result tables to: {RESULTS_DIR.resolve()}")
display(comparison[[
    "model", "n_features", "validation_r2", "test_r2", "test_mae", "test_rmse", "test_mape_pct",
    "test_r2_vs_baseline",
]])